In [ ]:
import itertools
import json
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.datasets import dataset_ref
from uavloc.filtering.ins_error import InsErrorPrior
from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.arms import FilterParams, build_correlator, estimator_factory
from uavloc.simulation.sequence import (all_fixes, flight_metrics, load_flight_set, per_flight_table, run_flights,
                                        village_bootstrap)
from uavloc.track import REPO_ROOT, experiment_dir, load_runs, log_run, set_seed
from uavloc.geo import LatLon, offset_latlon
from uavloc.matching.correlator import CorrelatorConfig, PositionCorrelator
from uavloc.simulation.camera import CameraConfig, RasterCamera
from uavloc.simulation.ins import InsConfig, InsSimulator
from uavloc.simulation.route import sample_route
from uavloc.simulation.sequence import fly, truth_in_ellipse
from uavloc.simulation.synthetic import ColourClassFeatures, class_world
from uavloc.data.rasters import load_raster

# Params

In [ ]:
EXP = "028-ins-error-state"
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
INITIAL_ERROR_M = {"normal": 0.0, "cold": 40.0}
SIGMA0_M = {"normal": 1.0, "cold": 25.0}
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
N_PARTICLES = 2000
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
TUNING_DIR = "artifacts/tuning"
BLIND_FLIGHTS = 6
BLIND_INSIDE_MIN = 0.95
SELF_MATCH_INITIAL_ERROR_M = 10.0
SELF_MATCH_FIX = 3
CONVERGED_M = 1.0
MAX_FIX_MS = 1000.0
SYNTHETIC_MPP = 0.385
SYNTHETIC_SIZE_PX = 6000
CHECK_TEMPERATURE = 0.5
CHECK_Q_POS = 0.1

In [ ]:
set_seed(SEED)
checks = {}

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split="tuning")
prior = InsErrorPrior(set_config.ins.velocity_bias_mps, set_config.ins.velocity_random_walk_mps, set_config.ins.heading_bias_deg, set_config.ins.speed_scale_std)

# Model

In [ ]:
def filter_params(temperature: float, q_pos: float) -> FilterParams:
    return FilterParams(temperature, q_pos, N_PARTICLES, 0, 0, 0.0, 0.0, 0.0, 0.0, WINDOW_CAP_HALF_M, WINDOW_MARGIN_M)

In [ ]:
def correlator_for(map_raster):
    return build_correlator(map_raster, CORRELATOR, DEVICE)

In [ ]:
def make_filter(correlator, seed: int):
    return estimator_factory("ins-error", filter_params(CHECK_TEMPERATURE, CHECK_Q_POS), ins_prior=prior)(correlator, specs[seed % len(specs)])

# Run

In [ ]:
center = LatLon(49.0, 25.0)
colours, probabilities = class_world(np.zeros((SYNTHETIC_SIZE_PX, SYNTHETIC_SIZE_PX), np.uint8), center, SYNTHETIC_MPP)
blind_correlator = PositionCorrelator(ColourClassFeatures(probabilities, SYNTHETIC_MPP),
                                      CorrelatorConfig(1.7, 60.0, sat_feature_crop_pct=0.03, matching="weighted-ncc",
                                                       class_weights=tuple(CORRELATOR["class_weights"])))
blind = []
for spec in specs[:BLIND_FLIGHTS]:
    shift = np.asarray(spec.waypoints).mean(0) - np.asarray(center)   # the route, moved onto the blank world
    poses = sample_route([LatLon(*(np.asarray(w) - shift)) for w in spec.waypoints], set_config.speed_mps,
                         set_config.step_s, set_config.altitude_m)
    track, fixes = fly(poses, RasterCamera(colours, CameraConfig()), InsSimulator(set_config.ins, spec.seed),
                       make_filter(blind_correlator, spec.seed), set_config.correction_interval_s, 1.0)
    inside = [truth_in_ellipse(LatLon(r.est_lat, r.est_lon), np.array([[r.cov_ee, r.cov_en], [r.cov_en, r.cov_nn]]),
                               LatLon(r.true_lat, r.true_lon)) for r in track.itertuples()]
    blind.append({"village": spec.village, "route": spec.route_index, "all_blind": bool((fixes["decision"] == "blind").all()),
                  "end_ins_error_m": float(track["ins_error_m"].iloc[-1]), "truth_in_99_ellipse": float(np.mean(inside))})
blind = pd.DataFrame(blind)
checks["blind_flights_consistent"] = {"min_inside": float(blind["truth_in_99_ellipse"].min()),
                                      "mean_inside": float(blind["truth_in_99_ellipse"].mean()),
                                      "pass": bool(blind["all_blind"].all() and blind["truth_in_99_ellipse"].mean() >= BLIND_INSIDE_MIN)}
blind

In [ ]:
spec = specs[0]
drone_raster = load_raster({"kind": "multidate", "place": spec.village, "date": set_config.drone_date})
self_correlator = correlator_for(drone_raster)
poses = sample_route(spec.waypoints, set_config.speed_mps, set_config.step_s, set_config.altitude_m)
exact = InsConfig(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, SELF_MATCH_INITIAL_ERROR_M)
_, self_fixes = fly(poses, RasterCamera(drone_raster, CameraConfig(set_config.fov_w_deg)), InsSimulator(exact, spec.seed),
                    make_filter(self_correlator, spec.seed), set_config.correction_interval_s, SELF_MATCH_INITIAL_ERROR_M)
checks["self_match_converges_exact_pose"] = {"error_after_fix_m": float(self_fixes["error_after_m"].iloc[SELF_MATCH_FIX - 1]),
                                             "pass": bool(self_fixes["error_after_m"].iloc[SELF_MATCH_FIX - 1] <= CONVERGED_M)}
checks["timing"] = {"median_fix_ms": float(self_fixes["fix_ms"].median()),
                    "pass": bool(self_fixes["fix_ms"].median() <= MAX_FIX_MS)}
checks

# Plots

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(self_fixes["t_s"], self_fixes["error_after_m"], marker="o")
ax.axhline(CONVERGED_M, color="grey", ls="--")
ax.set(xlabel="t, s", ylabel="error after fix, m", yscale="log", title="028 self-match, exact pose, 10 m initial error")
fig.tight_layout()

In [ ]:
Path("artifacts").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)
Path("artifacts/checks.json").write_text(json.dumps(checks, indent=1, default=float))
fig.savefig("figures/checks.png", dpi=110)
pd.Series({name: value["pass"] for name, value in checks.items()})